# 32. LoRA와 QLoRA — 적은 자원으로 파인튜닝

> **제32장** · **이론편 대응: 22.3~22.5절 (PEFT, LoRA, QLoRA)**
> **예상 소요**: 80분 (학습 2~3분)
> **필요 사양**: **[CPU]** 로 실행 가능 (GPU 있으면 더 큰 모델 가능)
> **추가 설치**: **peft** (1절 참조)
> **다운로드**: DistilGPT-2 (31장에서 받았다면 재사용)

---

## 이 장에서 하는 일

31장 6절에서 **전체 파인튜닝은 가중치의 4배 메모리가 필요**하다는 것을 봤다.
7B 모델이면 52GB다. 일반 GPU로는 불가능하다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **준비 — peft 설치** | — |
| 2 | LoRA의 아이디어 | 22.4절 |
| 3 | **파라미터 비율 0.39% 검증** ★ | 22.4절 |
| 4 | LoRA 직접 구현 | 22.4절 |
| 5 | peft로 적용하고 학습 | 22.4절 |
| 6 | **전체 파인튜닝과 비교** | 22.5절 |
| 7 | 어댑터 저장과 교체 | 22.4절 |
| 8 | 양자화와 QLoRA | 22.5절 |
| 9 | 8GB GPU에서 무엇이 가능한가 | 22.5절 |

---

## 1. 준비 — peft 설치

### PEFT란

**Parameter-Efficient Fine-Tuning** — 파라미터를 조금만 바꿔 미세조정하는 방법들의 총칭이다.
이론편 22.3절에서 다룬 개념이며, Hugging Face가 같은 이름의 라이브러리를 제공한다.

```
pip install peft
```

`accelerate`도 함께 설치된다. 여러 장치에 나눠 실행하는 것을 돕는 라이브러리다.

### PEFT에 포함된 기법들

| 기법 | 방식 |
|---|---|
| **LoRA** | 저차원 행렬 두 개를 곁들여 학습 (이 장) |
| Prefix Tuning | 입력 앞에 학습 가능한 벡터를 붙임 |
| Prompt Tuning | 프롬프트 임베딩만 학습 |
| IA³ | 활성값에 곱할 스케일만 학습 |

**LoRA가 가장 널리 쓰인다.** 성능 손실이 적으면서 구현이 단순하기 때문이다.

In [ ]:
import importlib

print("=" * 60)
print("필요 패키지 확인")
print("=" * 60)

required = [
    ("peft", "파라미터 효율적 미세조정", "pip install peft"),
    ("transformers", "모델 (23장)", "pip install transformers"),
    ("torch", "PyTorch", "pip install torch"),
]

missing = []
for name, desc, install in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "설치됨")
        print(f"[OK]   {name:<18}{ver:<14}{desc}")
    except ImportError:
        print(f"[없음] {name:<18}{'':<14}{desc}")
        missing.append(install)

# 양자화용 (선택)
print()
try:
    import bitsandbytes
    print(f"[선택] bitsandbytes {bitsandbytes.__version__} — QLoRA 4비트 양자화")
except ImportError:
    print("[선택] bitsandbytes 미설치 — 8절에서 개념만 다룹니다")
    print("       GPU가 있다면: pip install bitsandbytes")

print("-" * 60)
if missing:
    print("설치가 필요합니다:")
    for cmd in set(missing):
        print(f"  {cmd}")
else:
    print("[준비 완료] 2절로 진행하세요.")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import time

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print(f"모델: {MODEL_NAME}")

---

## 2. LoRA의 아이디어 — 이론편 22.4절

전체 파인튜닝은 가중치 행렬 $W$ 전체를 갱신한다. LoRA는 다르다.

**원본은 그대로 두고, 변화량만 따로 학습한다.**

$$W' = W + \Delta W$$

여기서 $\Delta W$를 통째로 학습하면 원본과 크기가 같아 절약이 안 된다.
**핵심은 $\Delta W$를 두 개의 작은 행렬로 쪼개는 것**이다.

$$\Delta W = BA, \qquad B \in \mathbb{R}^{d\times r},\ A \in \mathbb{R}^{r\times d}$$

$r$이 작으면($r \ll d$) 파라미터가 크게 줄어든다.

| | 크기 | 파라미터 수 |
|---|---|---|
| 원본 $W$ | $d \times d$ | $d^2$ |
| $A$ | $r \times d$ | $rd$ |
| $B$ | $d \times r$ | $dr$ |
| **합계** | | $2rd$ |

이론편 4.4절에서 다룬 **저차원 근사**가 여기서 쓰인다. 큰 행렬을 작은 것들의 곱으로 표현하는 것이다.

In [ ]:
import numpy as np

print("=" * 70)
print("행렬 크기 비교")
print("=" * 70)

d = 4096
print(f"원본 행렬: {d} x {d}")
print()
print(f"{'랭크 r':<10}{'A 크기':<16}{'B 크기':<16}{'합계':<16}{'원본 대비'}")
print("-" * 70)
for r in [4, 8, 16, 64]:
    a_size = r * d
    b_size = d * r
    total = a_size + b_size
    print(f"{r:<10}{f'{r}x{d}':<16}{f'{d}x{r}':<16}{total:<16,}{total/(d*d)*100:.2f}%")
print("-" * 70)
print(f"원본: {d*d:,}개")
print()
print("r이 두 배가 되면 파라미터도 정확히 두 배다 — 2rd 에 비례하기 때문이다.")

---

## 3. 파라미터 비율 검증 — 이론편 22.4절 값 ★

이론편 22.4절에서 계산한 표를 확인한다.

| 랭크 $r$ | LoRA 파라미터 | 전체 대비 | 절감 배수 |
|---|---|---|---|
| 4 | 32,768 | 0.20% | 512배 |
| **8** | **65,536** | **0.39%** | **256배** |
| 16 | 131,072 | 0.78% | 128배 |
| 64 | 524,288 | 3.12% | 32배 |

In [ ]:
import numpy as np

D_MODEL = 4096

print("=" * 70)
print("이론편 22.4절 값 검증")
print("=" * 70)
print(f"기준: {D_MODEL} x {D_MODEL} 가중치 행렬 하나")
print()

full_params = D_MODEL * D_MODEL
book = {4: (32768, 0.20, 512), 8: (65536, 0.39, 256),
        16: (131072, 0.78, 128), 64: (524288, 3.12, 32)}

print(f"{'r':<8}{'계산값':<16}{'이론편':<16}{'비율':<12}{'이론편':<10}{'절감':<10}{'이론편'}")
print("-" * 70)
for r in [4, 8, 16, 64]:
    lora = 2 * D_MODEL * r
    ratio = lora / full_params * 100
    save = full_params // lora
    b_lora, b_ratio, b_save = book[r]
    print(f"{r:<8}{lora:<16,}{b_lora:<16,}{ratio:<12.2f}{b_ratio:<10.2f}{save:<10}{b_save}")
    assert lora == b_lora, f"r={r} 파라미터 수가 다릅니다"
    assert abs(ratio - b_ratio) < 0.01
    assert save == b_save

print("-" * 70)
print(f"원본 파라미터: {full_params:,}")
print()
print("[OK] 이론편 22.4절 표와 완전히 일치")

In [ ]:
import numpy as np

print("=" * 70)
print("모델 전체로 확장하면 (이론편 22.4절)")
print("=" * 70)

# 70억 파라미터 모델 가정
n_layers = 32
n_matrices = 4          # Q, K, V, O
r = 8

lora_total = n_layers * n_matrices * (2 * D_MODEL * r)

print(f"가정: 32개 층, 층마다 Q·K·V·O 네 행렬, r=8")
print()
print(f"  {n_layers} x {n_matrices} x (2 x {D_MODEL} x {r})")
print(f"  = {lora_total:,}")
print()
print(f"약 {lora_total/1e6:.0f}만 개")
print(f"70억 파라미터 대비: {lora_total/7e9*100:.3f}%")
print()
print("이론편 22.4절: 약 839만 개, 0.12%")
assert abs(lora_total - 8_388_608) < 1000
print("[OK] 일치")
print()
print("100장이 넘는 책 전체를 다시 쓰는 대신")
print("몇 페이지에 메모지를 붙이는 정도의 작업이다.")

---

## 4. LoRA 직접 구현 — 이론편 22.4절

라이브러리를 쓰기 전에 **직접 만들어 본다.** 구조가 단순해 40줄이면 된다.

**중요한 세부사항 두 가지**

1. **초기화**: $A$는 무작위, $B$는 **0**으로 시작한다.
   그래야 $BA = 0$이 되어 처음에는 원본과 똑같이 동작한다.
2. **스케일링**: $\alpha/r$을 곱한다. $r$을 바꿔도 학습률을 다시 맞출 필요가 없게 하는 장치다.

In [ ]:
import torch
import torch.nn as nn
import math


class LoRALinear(nn.Module):
    # 기존 Linear 층에 LoRA를 곁들인다 (이론편 22.4절)

    def __init__(self, base_layer, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base_layer
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r        # 스케일링 계수

        in_features = base_layer.in_features
        out_features = base_layer.out_features

        # 원본 가중치는 얼린다
        for p in self.base.parameters():
            p.requires_grad = False

        # LoRA 행렬 두 개
        self.lora_A = nn.Parameter(torch.zeros(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # 초기화: A는 무작위, B는 0
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return base_out + lora_out * self.scaling


print("=" * 70)
print("LoRA 층 만들기")
print("=" * 70)

base = nn.Linear(768, 768)
lora_layer = LoRALinear(base, r=8, alpha=16)

n_base = sum(p.numel() for p in lora_layer.base.parameters())
n_lora = lora_layer.lora_A.numel() + lora_layer.lora_B.numel()
n_trainable = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)

print(f"원본 층      : {n_base:,}개 (얼림)")
print(f"LoRA A       : {lora_layer.lora_A.shape} = {lora_layer.lora_A.numel():,}")
print(f"LoRA B       : {lora_layer.lora_B.shape} = {lora_layer.lora_B.numel():,}")
print(f"학습 대상    : {n_trainable:,}개")
print(f"비율         : {n_trainable/n_base*100:.2f}%")
print()
print(f"스케일링 계수: alpha/r = {lora_layer.alpha}/{lora_layer.r} = {lora_layer.scaling}")

In [ ]:
import torch

print("=" * 70)
print("B를 0으로 초기화하는 이유")
print("=" * 70)

x = torch.randn(2, 768)

with torch.no_grad():
    out_base = lora_layer.base(x)
    out_lora = lora_layer(x)

print(f"원본 층 출력   : {out_base[0][:4].numpy().round(6)}")
print(f"LoRA 적용 출력 : {out_lora[0][:4].numpy().round(6)}")
print(f"차이           : {(out_lora - out_base).abs().max().item():.10f}")
print()
assert torch.allclose(out_base, out_lora, atol=1e-6)
print("[OK] 학습 전에는 원본과 완전히 같다")
print()
print("이것이 중요한 이유")
print("  LoRA를 붙였다는 것만으로 모델 성능이 떨어지면 안 된다.")
print("  B=0 이면 BA=0 이므로 아무 영향이 없다.")
print("  학습이 진행되면서 B가 조금씩 변해 원본을 조정하게 된다.")
print()

# B에 값을 넣어 확인
with torch.no_grad():
    lora_layer.lora_B.normal_(0, 0.01)
    out_after = lora_layer(x)
print(f"B에 값을 넣은 후 차이: {(out_after - out_base).abs().max().item():.6f}")
print("  → 이제 출력이 달라진다")

# 원복
with torch.no_grad():
    lora_layer.lora_B.zero_()

In [ ]:
import torch

print("=" * 70)
print("alpha/r 스케일링의 역할")
print("=" * 70)
print()
print("LoRA 출력에 alpha/r 을 곱한다.")
print("  r 을 키우면 BA 의 값이 커지는 경향이 있는데,")
print("  alpha/r 로 나눠 주면 r 이 달라져도 비슷한 크기를 유지한다.")
print()
print("따라서 r 을 바꿔도 학습률을 다시 조정할 필요가 줄어든다.")
print()

print(f"{'r':<8}{'alpha':<10}{'스케일링':<14}{'설명'}")
print("-" * 70)
for r, alpha in [(4, 16), (8, 16), (16, 16), (8, 8), (8, 32)]:
    note = ""
    if alpha == 2 * r:
        note = "관행적 설정 (alpha = 2r)"
    print(f"{r:<8}{alpha:<10}{alpha/r:<14.2f}{note}")
print("-" * 70)
print()
print("실무에서는 alpha = 2r 로 두는 경우가 많다.")
print("  r=8 이면 alpha=16, r=16 이면 alpha=32")

---

## 5. peft로 적용하고 학습 — 이론편 22.4절

직접 구현해 원리를 확인했으니, 이제 라이브러리를 쓴다.
**어느 층에 LoRA를 붙일지 지정하는 것**이 핵심 설정이다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

print("=" * 70)
print("peft로 LoRA 적용")
print("=" * 70)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
n_base_total = sum(p.numel() for p in base_model.parameters())

# ── LoraConfig 파라미터 ──────────────────────────────────────

#   task_type       작업 유형.  TaskType.CAUSAL_LM(생성)

#                   SEQ_CLS(분류) / SEQ_2_SEQ_LM / TOKEN_CLS

#   r               랭크.  기본값 8

#                   파라미터 수가 r 에 비례한다

#                   예: 4(가벼움) / 8(기본) / 16~64(표현력 필요)

#   lora_alpha      스케일링 계수.  기본값 8

#                   실제 스케일 = lora_alpha / r

#                   관행: alpha = 2r  (r=8 이면 16)

#   lora_dropout    드롭아웃.  기본값 0.0

#                   예: 0.05~0.1 — 데이터가 적을 때

#   target_modules  적용할 층 이름.  기본값 None(모델별 자동)

#                   GPT-2: ['c_attn']

#                   LLaMA: ['q_proj','v_proj'] 또는 전체 projection

#                   많이 지정할수록 표현력↑ 파라미터↑

#   bias            편향 학습 여부.  기본값 'none'

#                   'none' / 'all' / 'lora_only'

#   modules_to_save 추가로 학습할 층.  예: ['classifier']

# ──────────────────────────────────────────────────────────────

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                          # 랭크
    lora_alpha=16,                # 스케일링 (alpha/r = 2)
    lora_dropout=0.05,
    target_modules=["c_attn"],    # Attention의 Q·K·V 합친 층 (23장 5절)
)

# ── get_peft_model() ─────────────────────────────────────────

#   model    원본 모델

#   config   LoraConfig 등 PEFT 설정

#

#   하는 일

#     1) 원본 파라미터를 전부 requires_grad=False 로 얼린다

#     2) target_modules 에 지정한 층에 A·B 행렬을 붙인다

#     3) 그 행렬만 학습 대상이 된다

#

#   확인: model.print_trainable_parameters()

#         → 학습 대상이 전체의 몇 %인지 출력

# ──────────────────────────────────────────────────────────────

lora_model = get_peft_model(base_model, lora_config).to(device)

print()
lora_model.print_trainable_parameters()
print()

n_trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in lora_model.parameters())

print(f"원본 모델    : {n_base_total:,}개")
print(f"LoRA 추가 후 : {n_total:,}개")
print(f"학습 대상    : {n_trainable:,}개 ({n_trainable/n_total*100:.4f}%)")
print()
print("target_modules 지정")
print("  'c_attn' 은 GPT-2 에서 Q·K·V 를 합쳐 계산하는 층이다 (23장 5절).")
print("  모델마다 층 이름이 다르므로 확인이 필요하다.")

In [ ]:
import torch

print("=" * 70)
print("LoRA가 붙은 위치")
print("=" * 70)

lora_layers = []
for name, module in lora_model.named_modules():
    if name.endswith("lora_A.default"):
        lora_layers.append((name, tuple(module.weight.shape)))

print(f"LoRA 층 개수: {len(lora_layers)}")
print()
for name, shape in lora_layers[:3]:
    short = name.replace("base_model.model.transformer.", "")
    print(f"  {short:<44}{shape}")
if len(lora_layers) > 3:
    print(f"  ... (총 {len(lora_layers)}개)")
print()

# 얼린 파라미터와 학습 대상 확인
frozen = sum(p.numel() for p in lora_model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

print(f"{'구분':<20}{'파라미터':<16}{'비율'}")
print("-" * 70)
print(f"{'얼림 (원본)':<20}{frozen:<16,}{frozen/(frozen+trainable)*100:.2f}%")
print(f"{'학습 대상 (LoRA)':<20}{trainable:<16,}{trainable/(frozen+trainable)*100:.2f}%")
print("-" * 70)
print()
print("원본 가중치는 건드리지 않는다 — 이것이 LoRA의 핵심이다.")

In [ ]:
import torch
import time

# 31장과 같은 학습 데이터
train_data = [
    ("What is Python?", "Python is a high-level programming language."),
    ("What is a variable?", "A variable is a named container for a value."),
    ("What is a function?", "A function is a reusable block of code."),
    ("What is a loop?", "A loop repeats a block of code."),
    ("What is a list?", "A list is an ordered collection of items."),
    ("What is machine learning?", "Machine learning learns patterns from data."),
]

PROMPT = "Q: {q}\nA:"


def prepare(q, a):
    # 31장 3절의 손실 마스킹 그대로
    p_ids = tokenizer.encode(PROMPT.format(q=q))
    c_ids = tokenizer.encode(f" {a}{tokenizer.eos_token}")
    return p_ids + c_ids, [-100] * len(p_ids) + c_ids


print("=" * 70)
print("LoRA 학습")
print("=" * 70)

optimizer = torch.optim.AdamW(
    [p for p in lora_model.parameters() if p.requires_grad], lr=1e-3)

EPOCHS = 10
lora_history = []
t0 = time.time()

for epoch in range(EPOCHS):
    lora_model.train()
    total = 0.0
    for q, a in train_data:
        ids, labels = prepare(q, a)
        optimizer.zero_grad()
        loss = lora_model(
            input_ids=torch.tensor([ids]).to(device),
            labels=torch.tensor([labels]).to(device)).loss
        loss.backward()
        optimizer.step()
        total += loss.item()

    avg = total / len(train_data)
    lora_history.append(avg)
    if (epoch + 1) % 2 == 0:
        print(f"  에폭 {epoch+1:2}: 손실 {avg:.4f}  ({time.time()-t0:.0f}초)")

lora_time = time.time() - t0
print("-" * 70)
print(f"소요 시간: {lora_time:.0f}초")
print(f"학습률 1e-3 — 전체 파인튜닝(5e-5)보다 크게 잡았다.")
print("  학습 대상이 적어 더 큰 학습률을 써도 안정적이다.")

---

## 6. 전체 파인튜닝과 비교 — 이론편 22.5절

같은 데이터로 전체 파인튜닝도 해서 **자원 사용량을 비교**한다.

In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM

print("=" * 70)
print("전체 파인튜닝 (비교용)")
print("=" * 70)

torch.manual_seed(0)
full_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
optimizer_full = torch.optim.AdamW(full_model.parameters(), lr=5e-5)

full_history = []
t0 = time.time()

for epoch in range(EPOCHS):
    full_model.train()
    total = 0.0
    for q, a in train_data:
        ids, labels = prepare(q, a)
        optimizer_full.zero_grad()
        loss = full_model(
            input_ids=torch.tensor([ids]).to(device),
            labels=torch.tensor([labels]).to(device)).loss
        loss.backward()
        optimizer_full.step()
        total += loss.item()
    full_history.append(total / len(train_data))

full_time = time.time() - t0
print(f"소요 시간: {full_time:.0f}초")
print(f"최종 손실: {full_history[-1]:.4f}")

In [ ]:
import torch

def optimizer_state_size(opt):
    # 옵티마이저가 들고 있는 상태의 크기(바이트)
    size = 0
    for group in opt.param_groups:
        for p in group["params"]:
            for k, v in opt.state.get(p, {}).items():
                if torch.is_tensor(v):
                    size += v.numel() * v.element_size()
    return size


opt_lora = optimizer_state_size(optimizer)
opt_full = optimizer_state_size(optimizer_full)

n_lora_train = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
n_full_train = sum(p.numel() for p in full_model.parameters())

print("=" * 78)
print("LoRA vs 전체 파인튜닝 (이론편 22.5절)")
print("=" * 78)
print(f"{'항목':<24}{'LoRA':<20}{'전체 파인튜닝':<20}{'차이'}")
print("-" * 78)
print(f"{'학습 파라미터':<24}{n_lora_train:<20,}{n_full_train:<20,}{n_full_train/n_lora_train:>8.0f}배")
print(f"{'옵티마이저 상태(MB)':<24}{opt_lora/1024**2:<20.2f}{opt_full/1024**2:<20.1f}{opt_full/max(opt_lora,1):>8.0f}배")
print(f"{'학습 시간(초)':<24}{lora_time:<20.0f}{full_time:<20.0f}{full_time/lora_time:>8.1f}배")
print(f"{'최종 손실':<24}{lora_history[-1]:<20.4f}{full_history[-1]:<20.4f}")
print("-" * 78)
print()
print("옵티마이저 상태 차이가 특히 크다.")
print("  Adam은 파라미터마다 상태 두 개를 들고 있는데 (이론편 11.1절),")
print("  LoRA는 학습 대상이 적으니 그만큼 적게 든다.")
print()
print("31장 6절에서 계산한 '가중치의 4배' 중")
print("  그래디언트와 옵티마이저 상태(합쳐서 3배)가 거의 사라진다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- 왼쪽: 학습 곡선 ---
ax = axes[0]
ax.plot(range(1, EPOCHS+1), lora_history, marker="o",
        label="LoRA", linewidth=2, color="#0D9488")
ax.plot(range(1, EPOCHS+1), full_history, marker="s",
        label="전체 파인튜닝", linewidth=2, color="#EA580C")
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("학습 곡선 비교")
ax.legend()
ax.grid(alpha=0.3)

# --- 오른쪽: 자원 비교 ---
ax = axes[1]
categories = ["학습\n파라미터", "옵티마이저\n상태", "학습\n시간"]
lora_vals = [n_lora_train, opt_lora, lora_time]
full_vals = [n_full_train, opt_full, full_time]
ratios = [l/f*100 for l, f in zip(lora_vals, full_vals)]

bars = ax.bar(categories, ratios, color=["#0D9488", "#0D9488", "#0D9488"])
for b, r in zip(bars, ratios):
    ax.text(b.get_x()+b.get_width()/2, r+1.5, f"{r:.1f}%",
            ha="center", fontsize=10)
ax.axhline(100, color="#EA580C", linestyle="--", linewidth=1.5)
ax.text(2.3, 102, "전체 파인튜닝 = 100%", fontsize=8, color="#EA580C", ha="right")
ax.set_ylabel("전체 파인튜닝 대비 (%)")
ax.set_title("자원 사용량")
ax.set_ylim(0, 120)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("손실은 비슷하게 떨어지는데 자원은 훨씬 적게 쓴다.")
print("이것이 이론편 22.4절에서 LoRA가 널리 쓰이는 이유로 든 것이다.")

---

## 7. 어댑터 저장과 교체 — 이론편 22.4절

LoRA의 또 다른 장점이 있다. **학습 결과가 아주 작다.**

원본 가중치는 그대로이므로, **LoRA 행렬만 저장하면 된다.**

In [ ]:
import os
import torch
from pathlib import Path

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
save_dir = root / "outputs" / "lora_adapter"
save_dir.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("어댑터 저장")
print("=" * 70)

lora_model.save_pretrained(str(save_dir))

files = list(save_dir.glob("*"))
total_size = sum(f.stat().st_size for f in files if f.is_file())

print(f"저장 위치: {save_dir}")
print()
for f in sorted(files):
    if f.is_file():
        print(f"  {f.name:<32}{f.stat().st_size/1024:>8.1f} KB")
print("-" * 70)
print(f"  {'합계':<32}{total_size/1024:>8.1f} KB")
print()

# 전체 모델을 저장한다면
full_size = sum(p.numel() * p.element_size() for p in full_model.parameters())
print(f"전체 모델 저장 시: {full_size/1024**2:.0f} MB")
print(f"차이: {full_size/total_size:.0f}배")
print()
print("이것이 뜻하는 것")
print("  용도별로 어댑터를 여러 개 만들어 두고 바꿔 끼울 수 있다.")
print("  원본 모델은 하나만 메모리에 두면 된다.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

print("=" * 70)
print("어댑터 불러오기")
print("=" * 70)

# 원본 모델을 새로 불러온 뒤 어댑터만 붙인다
fresh_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
loaded = PeftModel.from_pretrained(fresh_base, str(save_dir)).to(device)
loaded.eval()

print("불러오기 완료")
print()


def generate(model, question, max_new_tokens=25):
    prompt = PROMPT.format(q=question)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()


print("학습한 어댑터로 생성")
for q, a in train_data[:2]:
    print(f"\n질문: {q}")
    print(f"  응답: {generate(loaded, q)[:60]}")
    print(f"  정답: {a[:60]}")

print()
print("-" * 70)
print("원본 모델 + 어댑터 = 학습된 모델")
print("  어댑터 파일 하나로 미세조정 결과를 옮길 수 있다.")

In [ ]:
import torch

print("=" * 70)
print("어댑터 병합 (merge)")
print("=" * 70)
print()
print("추론 속도가 중요하면 LoRA 행렬을 원본에 합칠 수 있다.")
print()
print("  W' = W + BA x (alpha/r)")
print()
print("합치고 나면 LoRA 층이 사라져 원본과 같은 구조가 된다.")
print("  → 추론 시 추가 계산이 없어진다")
print()

merged = loaded.merge_and_unload()
n_merged = sum(p.numel() for p in merged.parameters())
n_original = sum(p.numel() for p in fresh_base.parameters())

print(f"병합 후 파라미터: {n_merged:,}")
print(f"원본 파라미터   : {n_original:,}")
print(f"같은가: {n_merged == n_original}")
print()

# 병합 후에도 학습 효과가 유지되는지
merged = merged.to(device).eval()
print("병합된 모델로 생성")
q = train_data[0][0]
print(f"  질문: {q}")
print(f"  응답: {generate(merged, q)[:60]}")
print()
print("-" * 70)
print("병합의 장단점")
print("  장점: 추론 시 추가 연산 없음")
print("  단점: 어댑터를 바꿔 낄 수 없게 됨 (원본이 덮어씌워짐)")
print()
print("여러 용도로 쓸 계획이면 병합하지 않고 어댑터로 두는 편이 낫다.")

---

## 8. 양자화와 QLoRA — 이론편 22.5절

LoRA로 그래디언트와 옵티마이저 상태는 줄였다. **하지만 원본 가중치는 그대로다.**

7B 모델이면 FP16으로 13GB다. LoRA를 써도 이만큼은 메모리에 올려야 한다.

**양자화**가 이 문제를 푼다. 이론편 22.5절에서 다룬 대로,
값을 더 적은 비트로 표현해 저장 공간을 줄인다.

In [ ]:
import numpy as np

print("=" * 70)
print("양자화의 원리 (이론편 22.5절)")
print("=" * 70)

# 실제 가중치 하나를 가져와 양자화해 본다
import torch
w = full_model.transformer.h[0].attn.c_attn.weight.detach().flatten()[:8].numpy()

print("원본 값 (FP32)")
print(f"  {w.round(6)}")
print()

def quantize(values, bits):
    # 단순 선형 양자화
    levels = 2 ** bits
    vmin, vmax = values.min(), values.max()
    scale = (vmax - vmin) / (levels - 1)
    q = np.round((values - vmin) / scale)
    dq = q * scale + vmin
    return q.astype(int), dq

print(f"{'비트':<8}{'표현 가능 단계':<18}{'복원값 (앞 4개)':<40}{'평균 오차'}")
print("-" * 70)
for bits in [8, 4, 2]:
    q, dq = quantize(w, bits)
    err = np.abs(w - dq).mean()
    print(f"{bits:<8}{2**bits:<18}{str(dq[:4].round(5)):<40}{err:.6f}")
print("-" * 70)
print()
print("비트를 줄이면 저장 공간이 줄지만 정밀도를 잃는다.")
print()
print("이론편 22.5절의 요지")
print("  16비트 → 4비트면 저장은 1/4로 줄고,")
print("  표현 단계는 65,536개에서 16개로 줄어든다.")

In [ ]:
print("=" * 70)
print("QLoRA — 양자화 + LoRA (이론편 22.5절)")
print("=" * 70)
print()
print("핵심 발상: **어디에서 정밀도를 포기할지 구분한다**")
print()
print(f"{'구성 요소':<24}{'정밀도':<14}{'이유'}")
print("-" * 70)
print(f"{'원본 가중치 (얼림)':<24}{'4비트':<14}{'학습하지 않고 읽기만 함'}")
print(f"{'LoRA 행렬 (학습)':<24}{'16비트':<14}{'실제로 갱신되므로 정밀해야 함'}")
print("-" * 70)
print()
print("원본은 학습 중 값이 바뀌지 않으므로, 정밀도를 낮춰도 영향이 제한적이다.")
print("반면 LoRA 부분은 조금씩 갱신되므로 정밀도가 중요하다.")
print()

# 메모리 계산
print("=" * 70)
print("7B 모델 기준 메모리 (이론편 22.5절)")
print("=" * 70)

n = 7e9
r = 8
lora_params = 32 * 4 * (2 * 4096 * r)

configs = [
    ("전체 파인튜닝 (FP16)", n*2 + n*2 + n*4, "가중치+그래디언트+Adam"),
    ("LoRA (FP16 가중치)",   n*2 + lora_params*12, "가중치 + 작은 어댑터"),
    ("QLoRA (4비트 가중치)", n*0.5 + lora_params*12, "양자화 가중치 + 어댑터"),
]

print(f"{'방식':<26}{'메모리(GB)':<16}{'구성'}")
print("-" * 70)
for name, bytes_total, desc in configs:
    gb = bytes_total / 1024**3
    print(f"{name:<26}{gb:<16.1f}{desc}")
print("-" * 70)
print()
print("전체 파인튜닝은 52GB, QLoRA는 4GB 남짓이다.")
print("여기에 활성값이 더해지지만, 8GB GPU에서도 시도해 볼 수 있는 수준이 된다.")

In [ ]:
print("=" * 70)
print("QLoRA 코드 형태")
print("=" * 70)
print()
print("GPU와 bitsandbytes 가 있으면 다음처럼 쓴다.")
print()
code_lines = [
    "from transformers import AutoModelForCausalLM, BitsAndBytesConfig",
    "from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training",
    "import torch",
    "",
    "# 1) 4비트 양자화 설정",
    "bnb_config = BitsAndBytesConfig(",
    "    load_in_4bit=True,",
    "    bnb_4bit_quant_type='nf4',              # 정규분포에 맞춘 4비트",
    "    bnb_4bit_compute_dtype=torch.bfloat16,  # 계산은 16비트로",
    "    bnb_4bit_use_double_quant=True,         # 양자화 상수도 양자화",
    ")",
    "",
    "# 2) 양자화된 모델 로드",
    "model = AutoModelForCausalLM.from_pretrained(",
    "    MODEL_NAME,",
    "    quantization_config=bnb_config,",
    "    device_map='auto',",
    ")",
    "",
    "# 3) 학습 준비 (그래디언트 체크포인팅 등)",
    "model = prepare_model_for_kbit_training(model)",
    "",
    "# 4) LoRA 적용 — 여기서부터는 5절과 같다",
    "lora_config = LoraConfig(",
    "    r=8, lora_alpha=16, lora_dropout=0.05,",
    "    target_modules=['q_proj', 'v_proj'],    # 모델마다 이름이 다름",
    "    task_type='CAUSAL_LM',",
    ")",
    "model = get_peft_model(model, lora_config)",
]
for line in code_lines:
    print("  " + line)
print()
print("-" * 70)
print("주요 설정 설명")
print()
print("  nf4                  정규분포 형태에 맞춘 4비트 표현")
print("                       가중치 분포가 대체로 정규분포이므로 효율적")
print()
print("  compute_dtype        저장은 4비트, 계산은 16비트로")
print("                       계산 정확도를 유지하기 위해")
print()
print("  double_quant         양자화에 쓰는 상수도 양자화")
print("                       추가로 조금 더 줄인다")
print()
print("주의: bitsandbytes 는 CUDA가 필요하다. CPU에서는 동작하지 않는다.")

---

## 9. 8GB GPU에서 무엇이 가능한가 — 이론편 22.5절

이 책의 기준 사양인 **8GB GPU**에서 실제로 무엇을 할 수 있는지 정리한다.

In [ ]:
import torch

print("=" * 78)
print("8GB GPU 기준 가능 범위 (이론편 22.5절)")
print("=" * 78)

VRAM = 8.0
USABLE = VRAM * 0.85      # 실사용 가능분

print(f"전체 VRAM: {VRAM}GB, 실사용 가능(추정): {USABLE:.1f}GB")
print()
print(f"{'모델':<10}{'추론(FP16)':<16}{'추론(INT4)':<16}{'QLoRA 학습':<16}{'전체 학습'}")
print("-" * 78)

for n, name in [(0.5e9, "0.5B"), (1.5e9, "1.5B"), (3e9, "3B"),
                (7e9, "7B"), (13e9, "13B")]:
    fp16 = n * 2 / 1024**3
    int4 = n * 0.5 / 1024**3
    qlora = int4 * 1.6 + 1.0          # 어댑터·활성값 여유
    full = n * 8 / 1024**3            # 가중치+그래디언트+Adam

    def mark(v):
        return "가능" if v < USABLE else "어려움"

    print(f"{name:<10}{mark(fp16):<16}{mark(int4):<16}{mark(qlora):<16}{mark(full)}")

print("-" * 78)
print()
print("결론")
print(f"  추론  : INT4 로 7B 까지 가능")
print(f"  학습  : QLoRA 로 7B 까지 시도 가능 (배치 1, 짧은 시퀀스)")
print(f"  전체 파인튜닝: 0.5B 정도가 한계")
print()
print("실제로는 배치 크기·시퀀스 길이에 따라 달라진다.")
print("  긴 문장을 다루면 활성값이 크게 늘어난다 (11장 참조).")

In [ ]:
print("=" * 70)
print("메모리가 부족할 때 쓰는 방법")
print("=" * 70)
print()
print(f"{'방법':<28}{'효과':<20}{'대가'}")
print("-" * 70)
methods = [
    ("배치 크기 줄이기",        "활성값 감소",      "학습 불안정 (이론편 11.6절)"),
    ("gradient accumulation",  "작은 배치로 큰 효과", "속도 저하"),
    ("gradient checkpointing", "활성값 크게 감소",   "계산량 30% 증가"),
    ("시퀀스 길이 줄이기",      "활성값 감소",      "긴 문맥 처리 불가"),
    ("더 작은 r 사용",         "어댑터 크기 감소",   "표현력 감소"),
    ("target_modules 줄이기",  "어댑터 수 감소",    "성능 저하 가능"),
]
for a, b, c in methods:
    print(f"{a:<28}{b:<20}{c}")
print("-" * 70)
print()
print("gradient checkpointing 이 특히 효과적이다.")
print("  11장에서 봤듯 역전파를 위해 중간값을 저장하는데,")
print("  이것을 버렸다가 필요할 때 다시 계산하는 방식이다.")
print("  메모리를 크게 아끼는 대신 계산이 늘어난다.")
print()
print("  model.gradient_checkpointing_enable()")

---

## 10. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **22.4** | **r=8일 때 0.39%, 256배 절감** | **일치** ✓ |
| 22.4 | r=4/16/64 비율 | 전부 일치 ✓ |
| 22.4 | 7B 전체 적용 시 839만 개 (0.12%) | 일치 ✓ |
| 22.5 | QLoRA 메모리 절감 | 계산 확인 ✓ |
| 22.5 | 8GB에서 QLoRA로 7B 가능 | 계산 확인 ✓ |

### LoRA의 핵심

```python
# 원본은 얼리고
for p in base.parameters():
    p.requires_grad = False

# 작은 행렬 두 개만 학습
lora_A = nn.Parameter(torch.zeros(r, in_features))    # 무작위 초기화
lora_B = nn.Parameter(torch.zeros(out_features, r))   # 0으로 초기화

# 출력 = 원본 + BA x (alpha/r)
out = base(x) + (x @ lora_A.T @ lora_B.T) * (alpha / r)
```

### 기억할 것

| 항목 | 요점 |
|---|---|
| B를 0으로 초기화 | 학습 전에는 원본과 동일하게 |
| alpha/r | r을 바꿔도 학습률 재조정 불필요 |
| 학습률 | LoRA는 더 크게 (1e-3 수준) |
| `target_modules` | 모델마다 층 이름이 다름 |
| 어댑터 크기 | 수백 KB — 바꿔 끼우기 쉬움 |
| 병합 | 추론 빨라지지만 교체 불가 |
| QLoRA | 원본은 4비트, 어댑터는 16비트 |
| 8GB 기준 | QLoRA로 7B까지 시도 가능 |

### 31장과 이어지는 지점

| 24번 (전체 파인튜닝) | 25번 (LoRA) |
|---|---|
| 메모리 = 가중치 x4 | 가중치 x1 + 약간 |
| 저장 = 전체 모델 | 어댑터만 (수백 KB) |
| 학습률 5e-5 | 1e-3 |
| 손실 마스킹 | **그대로 사용** |

**손실 마스킹은 방식과 무관하다.** 31장 3절에서 만든 것을 그대로 썼다.

### 다음 장

**33. 양자화 — 모델을 작게 만들기** — 이론편 22.5절.
7장에서 QLoRA를 다루며 양자화를 개념으로만 설명했다.
이번에는 직접 8비트·4비트로 값을 변환해 보고, 실제 모델의 퍼플렉서티가 비트 수에 따라 어떻게 무너지는지 측정한다.

### 절감 효과를 그림으로

3절과 6절의 수치를 한자리에 모아 본다. **숫자로는 실감하기 어려운 규모**다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- 왼쪽: 랭크에 따른 파라미터 비율 (이론편 22.4절) ---
ax = axes[0]
d = 4096
rs = [4, 8, 16, 32, 64, 128]
ratios = [2 * d * r / (d * d) * 100 for r in rs]
bars = ax.bar([str(r) for r in rs], ratios, color="#1E40AF")
for b, r, v in zip(bars, rs, ratios):
    ax.text(b.get_x() + b.get_width()/2, v + 0.15, f"{v:.2f}%",
            ha="center", fontsize=8)
ax.axhline(0.39, color="#DC2626", linestyle="--", linewidth=1.5)
ax.text(4.2, 0.6, "r=8 → 0.39%\n(이론편 22.4절)", fontsize=8, color="#DC2626")
ax.set_xlabel("랭크 r")
ax.set_ylabel("전체 대비 비율 (%)")
ax.set_title("LoRA 파라미터 비율")
ax.grid(axis="y", alpha=0.3)

# --- 가운데: 자원 절감 (로그) ---
ax = axes[1]
categories = ["학습\n파라미터", "옵티마이저\n상태", "저장\n크기"]
lora_vals = [n_lora_train, max(opt_lora, 1), total_size]
full_vals = [n_full_train, max(opt_full, 1), full_size]

x = np.arange(len(categories))
w = 0.36
ax.bar(x - w/2, full_vals, w, label="전체 파인튜닝", color="#DC2626")
ax.bar(x + w/2, lora_vals, w, label="LoRA", color="#0D9488")
for i, (f, l) in enumerate(zip(full_vals, lora_vals)):
    ax.text(i, max(f, l) * 2.2, f"{f/l:.0f}배", ha="center",
            fontsize=10, color="#1E40AF")
ax.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels(categories, fontsize=8)
ax.set_ylabel("크기 (로그 눈금)")
ax.set_title("자원 사용량 비교")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3, which="both")

# --- 오른쪽: 모델 크기별 학습 가능 여부 ---
ax = axes[2]
model_sizes = [0.5, 1.5, 3, 7, 13]
full_gb = [n * 8 / 1.024**3 / 1000 * 1.024**3 for n in model_sizes]
full_gb = [n * 8 for n in model_sizes]        # FP16 x4
qlora_gb = [n * 0.5 * 1.6 + 1.0 for n in model_sizes]

ax.plot(model_sizes, full_gb, marker="o", linewidth=2,
        color="#DC2626", label="전체 파인튜닝")
ax.plot(model_sizes, qlora_gb, marker="s", linewidth=2,
        color="#0D9488", label="QLoRA")
ax.axhline(6.8, color="#1E40AF", linestyle="--", linewidth=1.5)
ax.text(0.7, 8.5, "8GB GPU 실사용 한계", fontsize=8, color="#1E40AF")
ax.set_xlabel("모델 크기 (B)")
ax.set_ylabel("필요 메모리 (GB)")
ax.set_yscale("log")
ax.set_title("어디까지 학습 가능한가")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print("왼쪽: r 이 두 배면 파라미터도 두 배 (2rd 에 비례)")
print()
print("가운데: 세 항목 모두 수백 배 차이 (로그 눈금이라 격차가 압축돼 보인다)")
print()
print("오른쪽: 8GB 기준으로 전체 파인튜닝은 0.5B 가 한계지만")
print("        QLoRA 면 7B 까지 시도할 수 있다.")